# exp04 — Attention Analysis

**Hypothesis (H3):** Within stego generations, mean attention weight from acrostic token K
to previously encoded acrostic tokens {1..K-1} increases monotonically with K.
No such trend is observed at matched sentence-initial positions in open generations.

**Falsifier:** No monotonic growth with K → model re-reads the target word from the prompt
each time, not from previously encoded tokens. H3 rejected.

**Data:** `exp01/valid_pairs.json` (303 pairs, Llama-3.1-8B). No new generation required.

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))  # add HF_TOKEN in Colab Secrets (🔑 icon)

    if not os.path.exists('stego_CoT'):
        os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
    os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate scipy')

MODEL_ID   = 'meta-llama/Llama-3.1-8B-Instruct'
INPUT_DIR  = 'results/exp01'
OUTPUT_DIR = 'results/exp04'
N_MAX      = 80   # pairs to process (2 forward passes each); ~30 min on L4

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('cwd:', os.getcwd())

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer

with open(f'{INPUT_DIR}/valid_pairs.json') as f:
    all_pairs = json.load(f)

hi_fid = [p for p in all_pairs if p['fidelity'] == 1.0]
print(f'Total: {len(all_pairs)}, high-fidelity: {len(hi_fid)}')
print(f'Payload lengths: {sorted(set(len(p["payload"]) for p in hi_fid))}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',  # required for output_attentions=True
)
model.eval()
model.config.use_cache = False

n_layers = model.config.num_hidden_layers
n_heads  = model.config.num_attention_heads
print(f'Layers: {n_layers}, heads: {n_heads}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def find_sentence_starts(token_ids, plen):
    """
    Return absolute token positions of paragraph starts in the CoT.
    Text-based approach: decode full CoT, find double-newline boundaries,
    then map character positions back to token indices.
    Robust to BPE merges (e.g. '.\n\n' -> single token id=382).
    """
    cot_ids   = list(token_ids[plen:])
    full_text = tokenizer.decode(cot_ids, skip_special_tokens=True)

    char_starts = []
    i = 0
    while i < len(full_text) and full_text[i] == '\n':
        i += 1
    if i < len(full_text):
        char_starts.append(i)
    while i < len(full_text):
        if full_text[i] == '\n':
            j = i + 1
            while j < len(full_text) and full_text[j] == '\n':
                j += 1
            if j - i >= 2 and j < len(full_text):
                char_starts.append(j)
            i = max(i + 1, j)
        else:
            i += 1

    positions = []
    found     = 0
    prev_len  = 0
    for tok_idx in range(len(cot_ids)):
        curr_len = len(tokenizer.decode(cot_ids[:tok_idx + 1], skip_special_tokens=True))
        while found < len(char_starts) and prev_len <= char_starts[found] < curr_len:
            positions.append(plen + tok_idx)
            found += 1
        prev_len = curr_len
        if found >= len(char_starts):
            break

    return positions


ex    = hi_fid[0]
plen  = len(ex['payload'])
s_pos = find_sentence_starts(ex['stego_ids'], ex['stego_plen'])[:plen]
o_pos = find_sentence_starts(ex['open_ids'],  ex['open_plen'])[:plen]
print(f'Payload: {ex["payload"]} (len={plen})')
print('Stego first tokens:', [tokenizer.decode([ex['stego_ids'][p]]) for p in s_pos])
print('Open  first tokens:', [tokenizer.decode([ex['open_ids'][p]])  for p in o_pos])

In [ ]:
@torch.no_grad()
def attention_vs_k(token_ids, sent_positions):
    """
    For each sentence K >= 1, compute mean attention weight from sent_positions[K]
    to sent_positions[:K], averaged over all heads, per layer.

    Returns np.ndarray of shape (len(sent_positions), n_layers).
    Row 0 is zeros (no previous positions for the first sentence).
    """
    ids_t   = torch.tensor([token_ids], dtype=torch.long).to(model.device)
    outputs = model(ids_t, output_attentions=True)
    attentions = [a[0].cpu().float() for a in outputs.attentions]
    del outputs, ids_t
    torch.cuda.empty_cache()

    n_sent = len(sent_positions)
    result = np.zeros((n_sent, n_layers))

    for k in range(1, n_sent):
        pos_k = sent_positions[k]
        prev  = sent_positions[:k]
        for l, attn in enumerate(attentions):
            # attn: (n_heads, seq_len, seq_len); mean over heads and prev positions
            result[k, l] = attn[:, pos_k, prev].mean().item()

    return result


ex    = hi_fid[0]
s_pos = find_sentence_starts(ex['stego_ids'], ex['stego_plen'])[:len(ex['payload'])]
test  = attention_vs_k(ex['stego_ids'], s_pos)
print(f'Output shape: {test.shape}  expected: ({len(s_pos)}, {n_layers})')
print('Non-zero rows (K>=1):', (test.sum(axis=1) > 0).sum())
print('Sample values (K=1, layers 0-4):', test[1, :5])

In [ ]:
stego_curves = []
open_curves  = []
skipped      = 0

for pair in hi_fid[:N_MAX]:
    plen  = len(pair['payload'])
    s_pos = find_sentence_starts(pair['stego_ids'], pair['stego_plen'])[:plen]
    o_pos = find_sentence_starts(pair['open_ids'],  pair['open_plen'])[:plen]
    n_pos = min(len(s_pos), len(o_pos))

    if n_pos < 2:
        skipped += 1
        continue

    stego_curves.append(attention_vs_k(pair['stego_ids'], s_pos[:n_pos]))
    open_curves.append(attention_vs_k(pair['open_ids'],   o_pos[:n_pos]))

    n = len(stego_curves)
    if n % 10 == 0:
        print(f'Processed {n} pairs (skipped {skipped})')

print(f'\nDone: {len(stego_curves)} pairs, {skipped} skipped')

In [ ]:
max_k  = max(c.shape[0] for c in stego_curves)
k_vals = list(range(1, max_k))

def collect_by_k(curves, k):
    return np.array([c[k].mean() for c in curves if k < c.shape[0]])

s_means = [collect_by_k(stego_curves, k).mean() for k in k_vals]
s_sems  = [collect_by_k(stego_curves, k).std() / np.sqrt(len(collect_by_k(stego_curves, k))) for k in k_vals]
o_means = [collect_by_k(open_curves,  k).mean() for k in k_vals]
o_sems  = [collect_by_k(open_curves,  k).std()  / np.sqrt(len(collect_by_k(open_curves,  k))) for k in k_vals]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.errorbar(k_vals, s_means, yerr=s_sems, marker='o', label='stego', color='steelblue')
ax.errorbar(k_vals, o_means, yerr=o_sems, marker='s', label='open',  color='coral', linestyle='--')
ax.set_xlabel('Sentence index K')
ax.set_ylabel('Mean attention to positions {1..K-1}')
ax.set_title('H3: attention to previous acrostic positions vs K')
ax.legend()

heatmap = np.array([
    [np.mean([c[k, l] for c in stego_curves if k < c.shape[0]])
     for l in range(n_layers)]
    for k in k_vals
])

ax2 = axes[1]
im  = ax2.imshow(heatmap, aspect='auto', cmap='Blues', origin='lower')
ax2.set_xlabel('Layer')
ax2.set_ylabel('K (sentence index)')
ax2.set_yticks(range(len(k_vals)))
ax2.set_yticklabels(k_vals)
ax2.set_title('Stego attention per layer')
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/attention_vs_k.png', dpi=150)
plt.show()

In [ ]:
def per_sample_slopes(curves):
    slopes = []
    for c in curves:
        ks = np.arange(1, c.shape[0])
        if len(ks) < 2:
            continue
        vals  = c[1:].mean(axis=1)
        slope = np.polyfit(ks, vals, 1)[0]
        slopes.append(slope)
    return np.array(slopes)


s_slopes = per_sample_slopes(stego_curves)
o_slopes = per_sample_slopes(open_curves)

print('=== H3: is attention slope > 0? ===')
for label, slopes in [('STEGO', s_slopes), ('OPEN', o_slopes)]:
    if len(slopes) == 0:
        print(f'{label}: no data')
        continue
    t, p = stats.ttest_1samp(slopes, 0)
    sig  = '  **' if p < 0.05 and slopes.mean() > 0 else ''
    print(f'{label}: mean slope = {slopes.mean():.6f}, t = {t:.3f}, p = {p:.4f}{sig}')

# Does stego slope exceed open slope? (paired test, same pairs)
n_paired = min(len(s_slopes), len(o_slopes))
if n_paired >= 2:
    diff_slopes = s_slopes[:n_paired] - o_slopes[:n_paired]
    t_d, p_d = stats.ttest_1samp(diff_slopes, 0)
    sig_d = '  **' if p_d < 0.05 and diff_slopes.mean() > 0 else ''
    print(f'\nStego - open slope: mean = {diff_slopes.mean():.6f}, t = {t_d:.3f}, p = {p_d:.4f}{sig_d}')

# Elevation: is stego attention higher than open at each K?
print('\n=== Elevation test: stego > open at each K? ===')
for k in k_vals:
    diffs = [
        stego_curves[i][k].mean() - open_curves[i][k].mean()
        for i in range(len(stego_curves))
        if k < stego_curves[i].shape[0] and k < open_curves[i].shape[0]
    ]
    if len(diffs) < 2:
        continue
    t_k, p_k = stats.ttest_1samp(diffs, 0)
    sig_k = '  **' if p_k < 0.05 and np.mean(diffs) > 0 else ''
    print(f'K={k}: mean diff = {np.mean(diffs):.5f}, t = {t_k:.3f}, p = {p_k:.4f}{sig_k}')

In [ ]:
summary = {
    'model':            MODEL_ID,
    'n_pairs':          len(stego_curves),
    'n_skipped':        skipped,
    'k_vals':           k_vals,
    'stego_mean_by_k':  [round(float(v), 6) for v in s_means],
    'open_mean_by_k':   [round(float(v), 6) for v in o_means],
    'stego_slope_mean': round(float(s_slopes.mean()), 6) if len(s_slopes) > 0 else None,
    'open_slope_mean':  round(float(o_slopes.mean()), 6) if len(o_slopes) > 0 else None,
}

out_path = f'{OUTPUT_DIR}/exp04_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)

if IN_COLAB:
    from google.colab import drive
    import shutil, pathlib
    drive.mount('/content/drive')
    dst = pathlib.Path('/content/drive/MyDrive/stego_cot_results/exp04')
    dst.mkdir(parents=True, exist_ok=True)
    for fp in pathlib.Path(OUTPUT_DIR).iterdir():
        shutil.copy(fp, dst / fp.name)
        print(f'  -> Drive: {fp.name}')